# Nutri-Lookup Dataset Ingestion, Inspection & Quality Analysis

This notebook imports and thoroughly checks the **Nutri-Lookup** data from `mydrive/CAPSTONE/nutri-lookup`.

- **Google Colab Environment**: Automatically mounts Google Drive at `/content/drive` and connects to `/content/drive/MyDrive/CAPSTONE/nutri-lookup` (with case-insensitive fallback checks).
- **Local Environment**: Automatically falls back to `./mydrive/CAPSTONE/nutri-lookup` or `./mydrive/capstone/nutri-lookup`.
- **Smart Ingestion**: Dynamically discovers and loads tabular files (`.csv`, `.tsv`, `.json`, `.parquet`, `.xlsx`, `.txt`).
- **Data Quality & Exploratory Analysis**:
  - File inventory & size summary
  - Schema, column types, and memory footprint
  - Missing values & duplicate records check
  - Summary statistics for macro- and micronutrients (calories, protein, carbs, fat, fiber, sodium, etc.)
  - Anomaly detection (negative values, extreme outliers) and distribution plots

In [1]:
# Step 1: Install required dependencies
%pip install -q pandas numpy matplotlib seaborn

In [2]:
import os
import sys
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Step 2: Configure Destination Directory
try:
    from google.colab import drive
    print("Detected Google Colab environment. Mounting Google Drive...")
    drive.mount('/content/drive')
    
    colab_candidates = [
        "/content/drive/MyDrive/CAPSTONE/nutri-lookup",
        "/content/drive/MyDrive/capstone/nutri-lookup",
        "/content/drive/MyDrive/CAPSTONE/nutrilookup",
        "/content/drive/MyDrive/capstone/nutrilookup",
        "/content/drive/MyDrive/nutri-lookup",
        "/content/drive/MyDrive/CAPSTONE/Nutri-Lookup",
        "/content/drive/MyDrive/capstone/Nutri-Lookup"
    ]
    DATA_DIR = next((p for p in colab_candidates if os.path.exists(p)), colab_candidates[0])
except ImportError:
    print("Running in local environment.")
    local_candidates = [
        os.path.abspath("mydrive/CAPSTONE/nutri-lookup"),
        os.path.abspath("mydrive/capstone/nutri-lookup"),
        os.path.abspath("../mydrive/CAPSTONE/nutri-lookup"),
        os.path.abspath("../mydrive/capstone/nutri-lookup"),
        os.path.abspath("nutri-lookup")
    ]
    DATA_DIR = next((p for p in local_candidates if os.path.exists(p)), local_candidates[0])

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Target Directory: {DATA_DIR}")
print(f"Directory exists: {os.path.exists(DATA_DIR)}")

Mounted at /content/drive
Target Directory: /content/drive/MyDrive/capstone/nutri-lookup
Directory exists: True


In [3]:
# Step 3: Scan directory and list all available files
print(f"Scanning files in {DATA_DIR}...\n" + "=" * 60)

all_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if not f.startswith('.'):
            full_path = os.path.join(root, f)
            rel_path = os.path.relpath(full_path, DATA_DIR)
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            all_files.append((rel_path, full_path, size_mb))

if all_files:
    print(f"Found {len(all_files)} file(s):\n")
    for rel_path, full_path, size_mb in sorted(all_files):
        print(f"  • {rel_path:<45} ({size_mb:.2f} MB)")
else:
    print(f"Notice: No files currently found in '{DATA_DIR}'.")
    print("Ensure your files are uploaded to Google Drive under 'MyDrive/CAPSTONE/nutri-lookup'.")

Scanning files in /content/drive/MyDrive/capstone/nutri-lookup...
Found 5 file(s):

  • branded_foods.csv                             (738.24 MB)
  • food_nutrients.csv                            (675.81 MB)
  • food_portions.csv                             (2.15 MB)
  • foods.csv                                     (200.81 MB)
  • nutrients.csv                                 (0.01 MB)


In [4]:
# Step 4: Define Universal Tabular Loader Function
def load_data_file(file_path):
    """
    Intelligently loads data files supporting .csv, .tsv, .json, .parquet, .xlsx, .txt.
    Handles multiple delimiters, encodings, and common tabular formats.
    """
    ext = os.path.splitext(file_path)[1].lower()
    
    if ext == '.parquet':
        return pd.read_parquet(file_path)
    elif ext in ['.xlsx', '.xls']:
        return pd.read_excel(file_path)
    elif ext == '.json':
        try:
            return pd.read_json(file_path)
        except Exception:
            return pd.read_json(file_path, lines=True)
    elif ext in ['.csv', '.tsv', '.txt']:
        encodings = ['utf-8', 'latin1', 'cp1252', 'iso-8859-1']
        for enc in encodings:
            try:
                with open(file_path, 'r', encoding=enc, errors='replace') as fp:
                    first_lines = [fp.readline() for _ in range(3)]
                sample = ''.join(first_lines)
                sep = '\t' if '\t' in sample else (';' if sample.count(';') > sample.count(',') else ',')
                return pd.read_csv(file_path, sep=sep, encoding=enc, low_memory=False)
            except Exception:
                continue
        # Fallback python engine
        return pd.read_csv(file_path, sep=None, engine='python')
    else:
        raise ValueError(f"Unsupported file format: {ext}")

In [5]:
# Step 5: Ingest all discovered files into Pandas DataFrames
datasets = {}

for rel_path, full_path, size_mb in all_files:
    ext = os.path.splitext(full_path)[1].lower()
    if ext in ['.csv', '.tsv', '.txt', '.json', '.parquet', '.xlsx', '.xls']:
        key = os.path.splitext(rel_path)[0].replace('/', '__').replace('\\', '__')
        try:
            df = load_data_file(full_path)
            datasets[key] = df
            print(f"✓ Loaded: '{key}' | {df.shape[0]:,} rows x {df.shape[1]} columns ({size_mb:.2f} MB)")
        except Exception as e:
            print(f"✗ Error loading '{rel_path}': {e}")

print(f"\nTotal datasets successfully loaded: {len(datasets)}")

✓ Loaded: 'nutrients' | 477 rows x 3 columns (0.01 MB)
✓ Loaded: 'foods' | 2,101,279 rows x 7 columns (200.81 MB)
✓ Loaded: 'branded_foods' | 1,999,950 rows x 11 columns (738.24 MB)
✓ Loaded: 'food_portions' | 47,445 rows x 8 columns (2.15 MB)
✓ Loaded: 'food_nutrients' | 27,195,013 rows x 6 columns (675.81 MB)

Total datasets successfully loaded: 5


In [6]:
# Step 6: Detailed Inspection, Schema & Missing Values Audit
try:
    from IPython.display import display
except ImportError:
    display = print

if not datasets:
    print("No datasets currently loaded. Once files are in the target directory, re-run this step.")
else:
    for name, df in datasets.items():
        print("=" * 80)
        print(f"DATASET: {name}")
        print("=" * 80)
        print(f"• Shape:              {df.shape[0]:,} rows x {df.shape[1]} columns")
        print(f"• Total Cells:        {df.size:,}")
        print(f"• Duplicate Rows:     {df.duplicated().sum():,} ({(df.duplicated().mean()*100):.2f}%)")
        print(f"• Memory Usage:       {df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
        
        # Column overview table
        print(f"\n--- Column Schema & Types ---")
        col_info = pd.DataFrame({
            'Column': df.columns,
            'Data Type': df.dtypes.astype(str),
            'Non-Null Count': df.notnull().sum(),
            'Null Count': df.isnull().sum(),
            'Null %': (df.isnull().mean() * 100).round(2),
            'Unique Values': df.nunique()
        })
        display(col_info)
        
        # Head preview
        print("\n--- First 5 Rows ---")
        display(df.head())
        
        # Summary stats for numerical fields
        num_cols = df.select_dtypes(include=[np.number]).columns
        if len(num_cols) > 0:
            print("\n--- Summary Statistics (Numerical Features) ---")
            display(df.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])
        print("\n" + "#" * 80 + "\n")

DATASET: nutrients
• Shape:              477 rows x 3 columns
• Total Cells:        1,431
• Duplicate Rows:     0 (0.00%)
• Memory Usage:       0.06 MB

--- Column Schema & Types ---


,Column,Data Type,Non-Null Count,Null Count,Null %,Unique Values
nutrient_id,nutrient_id,int64,477,0,0.0,477
nutrient_name,nutrient_name,object,477,0,0.0,474
unit_name,unit_name,object,477,0,0.0,12



--- First 5 Rows ---


,nutrient_id,nutrient_name,unit_name
0,2047,Energy (Atwater General Factors),KCAL
1,2048,Energy (Atwater Specific Factors),KCAL
2,1001,Solids,G
3,1002,Nitrogen,G
4,1003,Protein,G



--- Summary Statistics (Numerical Features) ---


...


--- First 5 Rows ---


,fdc_id,nutrient_id,amount_per_100g,data_points,derivation_id,percent_daily_value
0,1105904,1257,0.00,NaN,71.0,NaN
1,1105904,1293,53.33,NaN,71.0,0.0
2,1105904,1253,0.00,NaN,75.0,0.0
3,1105904,1092,0.00,NaN,75.0,0.0
4,1105904,1008,867.00,NaN,71.0,NaN



--- Summary Statistics (Numerical Features) ---


,count,mean,std,min,25%,50%,75%,max
fdc_id,27195013.0,1.613944e+06,7.407122e+05,167512.000,1082639.0,1689883.00,2223146.0,2.768939e+06
nutrient_id,27195013.0,1.171744e+03,2.402806e+02,1002.000,1008.0,1093.00,1253.0,2.069000e+03
amount_per_100g,27195013.0,7.552167e+02,1.464427e+06,-0.705,0.0,3.66,40.0,7.500000e+09
data_points,810943.0,2.395882e+00,1.599906e+01,0.000,0.0,0.00,1.0,2.526000e+03
derivation_id,26642933.0,7.063753e+01,8.932605e+00,1.000,70.0,70.00,75.0,7.900000e+01
percent_daily_value,20385472.0,7.128944e+00,4.597629e+01,-4.000,0.0,2.00,9.0,3.333000e+04



################################################################################



In [7]:
# Step 7: Nutrition Feature Identification & Anomaly Checks
print("=== Nutrition-Specific Exploratory Analysis ===\n")

nutrient_keywords = [
    'calor', 'energ', 'protein', 'carb', 'fat', 'lipid', 'fiber', 'fibr',
    'sugar', 'sodiu', 'cholest', 'potass', 'calciu', 'iron', 'vitam', 'zinc', 'water'
]

for name, df in datasets.items():
    matched_cols = [c for c in df.columns if any(k in str(c).lower() for k in nutrient_keywords)]
    num_matched = [c for c in matched_cols if np.issubdtype(df[c].dtype, np.number)]
    
    if num_matched:
        print(f"[Dataset: {name}] - Detected {len(num_matched)} nutrient metric(s):")
        print(f"  Columns: {num_matched}\n")
        
        # Check for negative values
        anomalies = {c: int((df[c] < 0).sum()) for c in num_matched if (df[c] < 0).sum() > 0}
        if anomalies:
            print(f"  ⚠️ Warning: Found negative nutrient values: {anomalies}")
        else:
            print("  ✓ All nutrient values are non-negative (valid).")
            
        # Top 5 items for main macronutrients if item name exists
        name_cols = [c for c in df.columns if any(k in str(c).lower() for k in ['name', 'food', 'item', 'desc', 'recipe', 'title'])]
        if name_cols:
            label_col = name_cols[0]
            for metric in num_matched[:3]:
                top_items = df.nlargest(5, metric)[[label_col, metric]]
                print(f"\n  Top 5 items by '{metric}':")
                for _, r in top_items.iterrows():
                    print(f"    • {r[label_col]}: {r[metric]}")
        print("\n" + "-" * 60 + "\n")

=== Nutrition-Specific Exploratory Analysis ===



In [8]:
# Step 8: Visualizations of Key Nutritional Metric Distributions
sns.set_theme(style="whitegrid")

for name, df in datasets.items():
    matched_cols = [c for c in df.columns if any(k in str(c).lower() for k in nutrient_keywords)]
    num_matched = [c for c in matched_cols if np.issubdtype(df[c].dtype, np.number)]
    
    if len(num_matched) >= 2:
        cols_to_plot = num_matched[:4]
        fig, axes = plt.subplots(1, len(cols_to_plot), figsize=(5 * len(cols_to_plot), 4))
        if len(cols_to_plot) == 1:
            axes = [axes]
        
        for ax, col in zip(axes, cols_to_plot):
            sns.histplot(df[col].dropna(), kde=True, ax=ax, color='teal')
            ax.set_title(f"Distribution: {col}", fontsize=11, fontweight='bold')
            ax.set_xlabel(col)
            ax.set_ylabel("Count")
            
        plt.suptitle(f"Nutritional Metric Distributions ({name})", fontsize=13, y=1.03)
        plt.tight_layout()
        plt.show()
        print("\n")

In [9]:
# Step 9: Final Quality & Inventory Summary Table
summary_rows = []

for name, df in datasets.items():
    summary_rows.append({
        "Dataset": name,
        "Total Rows": f"{df.shape[0]:,}",
        "Total Columns": df.shape[1],
        "Null Cells (%)": f"{(df.isnull().sum().sum() / df.size * 100):.2f}%",
        "Duplicates": df.duplicated().sum(),
        "Numeric Features": len(df.select_dtypes(include=[np.number]).columns),
        "Categorical Features": len(df.select_dtypes(include=['object', 'category']).columns)
    })

if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    print("=== Nutri-Lookup Dataset Inventory & Health Summary ===")
    try:
        from IPython.display import display
        display(df_summary)
    except ImportError:
        print(df_summary.to_string())
else:
    print("No datasets were analyzed.")

=== Nutri-Lookup Dataset Inventory & Health Summary ===


,Dataset,Total Rows,Total Columns,Null Cells (%),Duplicates,Numeric Features,Categorical Features
0,nutrients,477,3,0.00%,0,1,2
1,foods,"2,101,279",7,28.04%,0,1,6
2,branded_foods,"1,999,950",11,20.31%,0,2,9
3,food_portions,"47,445",8,22.00%,359,5,3
4,food_nutrients,"27,195,013",6,20.68%,1,6,0
